# Reverse-engineering Polymarket wallet `0x20d2…e29d`
Source: <https://gravia.trade/0x20d2309cd92b797ae7ca175ed828ed8a27fbe29d>

The wallet trades Polymarket's short crypto **Up/Down** markets (BTC/ETH/SOL/XRP, 5m and 15m). Each market resolves on the Chainlink **TWAP-60s** stream: the TWAP at the end of the window has to be ≥ the TWAP at the start for "Up".

This notebook:
1. pulls the wallet's full fill history (data-api `/activity`, paged backward with `end`, so the 10k offset cap doesn't apply)
2. pulls Binance 1s klines as the reference price (daily archives plus the REST API for today)
3. pulls actual resolutions from Gamma
4. rebuilds the market state at every fill (lead vs. the open, short-term momentum, volatility, TWAP fair value) and looks at how the wallet decides what to buy, when, at what price, and how much

Everything is cached in `cache/`. Reruns only fetch the new data. Delete a file to force a refetch.

## Findings (run of 2026-10-29, 7-day lookback, 14,376 fills)
* **Universe:** only the BTC/ETH/SOL/XRP 5m markets and BTC 15m. About $237k turnover, **+$5.6k P&L at resolution** (2.6% of buy notional, before fees and rebates).
* **Reference price:** Binance's TWAP-60 proxy reproduces 98.3% of resolutions, vs. 88% for plain spot close against open. The resolution mechanics matter.
* **It prices off a fair-value model.** A depth-3 tree on the TWAP/Brownian-motion z-score explains **R² = 0.87** of the price it pays. In other words, it bids roughly Φ(z) on the side it buys.
* **The trigger is short-horizon Binance momentum (latency arbitrage).** The bought side's Binance move before the fill is +0.9 bps over 3s and +1.3 bps over 5s (t ≈ 62 and 78). Over 1s it's only +0.1 bps, which is consistent with a decision ~1–2s before the fill is mined. It buys after Binance ticks its way and before the Polymarket book reprices.
* **The edge is real where the model says it is.** Fills with model edge > 0 make about +$6.9k; fills with edge < 0 lose about −$2.1k. The best pocket is `mom_3s_side > 0.6 bps`, `price ≤ 0.78`, `z_side > 0.56` (+$0.13 per share). Buying expensive (>0.78) favourites on momentum loses money.
* **72% of buys are on the side currently ahead.** 35% of windows are traded on both sides, with VWAP(Up)+VWAP(Down) at a median of 1.10. That's re-entry after flips, not a cheap-pair arb. SELLs are centred on a 0 bps lead, so they look like inventory unwinds rather than stop-losses.
* **Sizing:** fixed share clips (5/10/20/30/60) that scale up for BTC and for higher-edge fills. First entry comes after about 70–130s in the 5m markets, and there are no fills in the first 8s.

In [ ]:
import io, json, time, zipfile, math, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, requests
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import ndtr

WALLET = "0x20d2309cd92b797ae7ca175ed828ed8a27fbe29d"
LOOKBACK_DAYS = 7            # how much history to analyse
TWAP_W = 60                  # Chainlink TWAP-60s stream
VOL_LOOKBACK = 300           # seconds of 1s returns for realised vol
CACHE = Path("cache") / WALLET[:10]; CACHE.mkdir(parents=True, exist_ok=True)
KL_CACHE = Path("cache") / "binance_1s"; KL_CACHE.mkdir(parents=True, exist_ok=True)

S = requests.Session(); S.headers["User-Agent"] = "Mozilla/5.0"
def get_json(url, params=None, tries=6):
    for i in range(tries):
        try:
            r = S.get(url, params=params, timeout=30)
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(1.5 * 2**i); continue
            r.raise_for_status(); return r.json()
        except requests.ConnectionError:
            time.sleep(1.5 * 2**i)
    raise RuntimeError(f"failed {url} {params}")
NOW = int(time.time()); SINCE = NOW - LOOKBACK_DAYS * 86400

## 1. Wallet activity (cached, incremental)

In [ ]:
ACT_F = CACHE / "activity.parquet"

def fetch_activity(start=None, end=None):
    rows, cur_end = [], end
    while True:
        p = dict(user=WALLET, limit=500, sortBy="TIMESTAMP", sortDirection="DESC")
        if cur_end is not None: p["end"] = cur_end
        if start is not None: p["start"] = start
        r = get_json("https://data-api.polymarket.com/activity", p)
        if not r: break
        rows += r
        mn = min(x["timestamp"] for x in r)
        if len(r) < 500 or (start is not None and mn <= start): break
        # step back; rows sharing the boundary second get re-pulled and deduped below
        cur_end = mn if mn != cur_end else mn - 1
    return pd.DataFrame(rows)

act = pd.read_parquet(ACT_F) if ACT_F.exists() else pd.DataFrame()
parts = [act]
if act.empty:
    parts.append(fetch_activity(start=SINCE))
else:
    parts.append(fetch_activity(start=int(act.timestamp.max())))          # newer
    if act.timestamp.min() > SINCE + 3600:
        parts.append(fetch_activity(start=SINCE, end=int(act.timestamp.min())))  # older
act = pd.concat(parts, ignore_index=True)
act = act.drop_duplicates(["transactionHash", "asset", "type", "side", "size", "price"]).sort_values("timestamp").reset_index(drop=True)
act.to_parquet(ACT_F)
print(len(act), pd.to_datetime(act.timestamp.min(), unit="s"), "->", pd.to_datetime(act.timestamp.max(), unit="s"))
act.type.value_counts()

In [ ]:
act = act[act.timestamp >= SINCE].copy()
tr = act[act.type == "TRADE"].copy()
m = tr.slug.str.extract(r"^(?P<coin>[a-z]+)-updown-(?P<dur>\d+)m-(?P<t0>\d{10})$")
print("fills outside the m-updown markets:", m.coin.isna().sum()); print(tr[m.coin.isna()].slug.str[:40].value_counts().head())
tr = tr.join(m).dropna(subset=["coin"])
tr["dur"] = tr.dur.astype(int) * 60; tr["t0"] = tr.t0.astype(int); tr["t1"] = tr.t0 + tr.dur
tr["elapsed"] = tr.timestamp - tr.t0; tr["remaining"] = tr.t1 - tr.timestamp
tr["mkt"] = tr.coin + "-" + (tr.dur // 60).astype(str) + "m"
tr["usd"] = tr.usdcSize
tr["ts"] = pd.to_datetime(tr.timestamp, unit="s")
tr.groupby(["mkt", "side"]).agg(fills=("usd", "size"), usd=("usd", "sum"), avg_px=("price", "mean")).round(3)

## 2. Actual resolutions (Gamma, cached per slug)

In [ ]:
RES_F = CACHE / "resolutions.parquet"
res = pd.read_parquet(RES_F) if RES_F.exists() else pd.DataFrame(columns=["slug", "up_won"])
todo = sorted(set(tr.loc[tr.t1 < NOW - 120, "slug"]) - set(res.slug))
new = []
for i in range(0, len(todo), 40):
    r = get_json("https://gamma-api.polymarket.com/markets", [("slug", s) for s in todo[i:i+40]] + [("limit", 100), ("closed", "true")])
    for mk in r:
        px_ = [float(x) for x in json.loads(mk["outcomePrices"])] if mk.get("outcomePrices") else None
        if mk.get("umaResolutionStatus") == "resolved" or (px_ and max(px_) == 1):
            new.append(dict(slug=mk["slug"], up_won=int(px_[0] == 1)))
if new:
    res = pd.concat([res, pd.DataFrame(new)]).drop_duplicates("slug"); res.to_parquet(RES_F)
print(len(res), "resolved markets cached;", len(todo) - len(new), "still unresolved / missing")

## 3. Binance 1s klines (daily zip archive + REST for recent days, cached)

In [ ]:
KCOLS = ["ot", "o", "h", "l", "c", "v", "ct", "qv", "n", "tbv", "tqv", "ig"]

def _norm(df):
    df = df.iloc[:, :6].copy(); df.columns = KCOLS[:6]
    ot = df.ot.astype("int64"); df["t"] = np.where(ot > 1e14, ot // 1_000_000, ot // 1000)  # archives switched to µs
    return df[["t", "o", "h", "l", "c", "v"]].astype({"o": float, "h": float, "l": float, "c": float, "v": float})

def kl_day(sym, day):
    f = KL_CACHE / f"{sym}_{day}.parquet"
    if f.exists(): return pd.read_parquet(f)
    url = f"https://data.binance.vision/data/spot/daily/klines/{sym}/1s/{sym}-1s-{day}.zip"
    r = S.get(url, timeout=120)
    if r.status_code == 200:
        z = zipfile.ZipFile(io.BytesIO(r.content)); df = _norm(pd.read_csv(z.open(z.namelist()[0]), header=None))
        df.to_parquet(f); return df
    return None   # archive not published yet -> REST

def kl_rest(sym, start, end):
    # hourly chunks; only fully-past hours get cached
    out = []
    for h0 in range(start - start % 3600, end, 3600):
        f = KL_CACHE / f"{sym}_h{h0}.parquet"
        if f.exists(): out.append(pd.read_parquet(f)); continue
        rows, s = [], h0 * 1000
        while s < (h0 + 3600) * 1000:
            r = get_json("https://data-api.binance.vision/api/v3/klines",
                         dict(symbol=sym, interval="1s", startTime=s, endTime=(h0 + 3600) * 1000 - 1, limit=1000))
            if not r: break
            rows += r; s = r[-1][0] + 1000
        df = _norm(pd.DataFrame(rows)) if rows else pd.DataFrame(columns=["t", "o", "h", "l", "c", "v"])
        if h0 + 3600 < time.time() - 60: df.to_parquet(f)
        out.append(df)
    return pd.concat(out)

def klines(sym, start, end):
    parts, d = [], dt.datetime.utcfromtimestamp(start).date()
    while d <= dt.datetime.utcfromtimestamp(end).date():
        k = kl_day(sym, d.isoformat())
        if k is None:
            ds = int(dt.datetime(d.year, d.month, d.day, tzinfo=dt.timezone.utc).timestamp())
            k = kl_rest(sym, max(ds, start), min(ds + 86400, end))
        parts.append(k); d += dt.timedelta(days=1)
    k = pd.concat(parts).drop_duplicates("t").set_index("t").sort_index()
    full = np.arange(k.index.min(), k.index.max() + 1)
    return k.reindex(full).assign(c=lambda x: x.c.ffill())   # forward-fill gaps (no trades in that second)

SYM = {c: f"{c.upper()}USDT" for c in tr.coin.unique()}
K = {c: klines(s, int(tr.t0.min()) - 600, int(tr.t1.max()) + 5) for c, s in SYM.items()}
{c: len(k) for c, k in K.items()}

## 4. State at each fill
For each fill, as of the last full second before the fill's block time:
* `ret_bps`: spot vs. the window's open price (from Binance, a proxy for Chainlink)
* `twap_lead_bps`: the current projected end TWAP-60 vs. the start TWAP-60, which is how the market actually resolves
* `mom_{k}s`: Binance return over the last k seconds
* `fair_up`: Up probability under Brownian motion with realised 1s vol, with the averaging of the final 60s handled exactly

All directional features are also **signed toward the outcome bought** (`*_side`), so >0 means "the side it bought is winning or moving its way".

In [ ]:
FEAT_F = CACHE / "features.parquet"

def state(coin, g):
    k = K[coin]; c = k.c; idx0 = c.index[0]; arr = c.values
    lr = np.diff(np.log(arr), prepend=np.log(arr[0]))
    vol = pd.Series(lr).rolling(VOL_LOOKBACK, min_periods=60).std().values
    cs = np.concatenate([[0], np.cumsum(arr)])              # prefix sums for fast TWAPs
    twap = lambda a, b: (cs[b - idx0] - cs[a - idx0]) / (b - a)   # mean close over seconds [a, b)
    out = []
    for t, t0, t1 in zip(g.timestamp.values, g.t0.values, g.t1.values):
        now = min(t, t1) - 1                                  # last completed second
        if now - idx0 < VOL_LOOKBACK or t1 - idx0 > len(arr): out.append({}); continue
        s, p0 = arr[now - idx0], arr[t0 - 1 - idx0]           # open = close of second before t0
        sig = vol[now - idx0] * s
        R = t1 - 1 - now                                      # seconds to go
        tw0 = twap(t0 - TWAP_W, t0)
        a = max(t1 - TWAP_W, now + 1) if R < TWAP_W else None
        if R >= TWAP_W:
            mu, var = s, sig**2 * ((R - TWAP_W) + TWAP_W / 3)
        else:
            known = cs[now + 1 - idx0] - cs[t1 - TWAP_W - idx0]
            mu, var = (known + s * R) / TWAP_W, sig**2 * R**3 / (3 * TWAP_W**2)
        z = (mu - tw0) / math.sqrt(var) if var > 0 else np.sign(mu - tw0) * 50
        mom = {f"mom_{w}s": (s / arr[now - w - idx0] - 1) * 1e4 for w in (1, 3, 5, 10, 30, 60)}
        out.append(dict(spot=s, p0=p0, ret_bps=(s / p0 - 1) * 1e4, twap_lead_bps=(mu / tw0 - 1) * 1e4,
                        vol_bps=vol[now - idx0] * 1e4, z=z, fair_up=ndtr(z),
                        end_spot_up=int(arr[t1 - 1 - idx0] >= p0), end_twap_up=int(twap(t1 - TWAP_W, t1) >= tw0), **mom))
    return pd.DataFrame(out, index=g.index)

cached = pd.read_parquet(FEAT_F) if FEAT_F.exists() else pd.DataFrame()
need = tr[~tr.transactionHash.isin(cached.get("transactionHash", pd.Series(dtype=str)))] if not cached.empty else tr
new = pd.concat([state(c, g) for c, g in need.groupby("coin")]) if len(need) else pd.DataFrame()
if len(new) and "spot" in new:   # fills in still-open windows have no state yet
    new = need[["transactionHash", "asset", "side", "size", "price"]].join(new).dropna(subset=["spot"])
    cached = pd.concat([cached, new]).drop_duplicates(["transactionHash", "asset", "side", "size", "price"]); cached.to_parquet(FEAT_F)
df = tr.merge(cached, on=["transactionHash", "asset", "side", "size", "price"], how="inner").merge(res, on="slug", how="left")
sgn = np.where(df.outcome == "Up", 1, -1)
for col in ["ret_bps", "twap_lead_bps", "z"] + [c for c in df if c.startswith("mom_")]:
    df[col + "_side"] = df[col] * sgn
df["fair_side"] = np.where(df.outcome == "Up", df.fair_up, 1 - df.fair_up)
df["won"] = np.where(df.outcome == "Up", df.up_won, 1 - df.up_won)
df["pnl"] = np.where(df.side == "BUY", 1, -1) * df["size"] * (df.won - df.price)
df["edge"] = np.where(df.side == "BUY", df.fair_side - df.price, df.price - df.fair_side)
print(len(df), "fills with features;", df.won.notna().sum(), "with resolved outcome")

### Sanity check: which Binance proxy matches the real resolution?

In [ ]:
chk = df.dropna(subset=["up_won"]).drop_duplicates("slug")
pd.Series({"spot close >= open": (chk.end_spot_up == chk.up_won).mean(),
           "TWAP60 end >= TWAP60 start": (chk.end_twap_up == chk.up_won).mean(), "n markets": len(chk)}).round(4)

## 5. P&L overview

In [ ]:
d = df.dropna(subset=["won"]).sort_values("timestamp")
print(f"fills {len(d)}  turnover ${d.usd.sum():,.0f}  P&L ${d.pnl.sum():,.0f}  ({d.pnl.sum()/d.usd[d.side=='BUY'].sum():.2%} of buy notional) — excludes fees/rebates")
fig = go.Figure()
for mk, g in d.groupby("mkt"):
    fig.add_scatter(x=g.ts, y=g.pnl.cumsum(), name=mk, mode="lines")
fig.add_scatter(x=d.ts, y=d.pnl.cumsum(), name="ALL", line=dict(width=3, color="black"))
fig.update_layout(title="Cumulative P&L at resolution (USDC)", height=450).show()
d.groupby("mkt").agg(fills=("pnl", "size"), usd=("usd", "sum"), pnl=("pnl", "sum"), win=("won", "mean"), avg_px=("price", "mean")).round(3)

## 6. When does it trade?

In [ ]:
b = df[df.side == "BUY"]
fig = px.histogram(df, x="elapsed", color="side", facet_col="dur", nbins=60, barmode="overlay",
                   title="Seconds into the window at fill (facets: window length)")
fig.update_xaxes(matches=None).show()
first = b.sort_values("timestamp").groupby("slug").first()
print("first entry, seconds into window:"); print(first.groupby("mkt").elapsed.describe().round(0))

## 7. What does it buy? Leader vs. laggard, momentum

In [ ]:
b = df[df.side == "BUY"].copy()
b["buys_leader"] = b.twap_lead_bps_side > 0
print("share of buys on the side currently ahead (TWAP basis):", round(b.buys_leader.mean(), 3))
mom_cols = [c for c in b if c.startswith("mom_") and c.endswith("_side")]
rng = np.random.default_rng(0)
print("mean Binance move toward the bought side before the fill (bps):"); print(b[mom_cols].mean().round(2))
print("t-stat:"); print((b[mom_cols].mean() / b[mom_cols].std() * np.sqrt(len(b))).round(1))

In [ ]:
fig = px.scatter(b.sample(min(len(b), 6000), random_state=0), x="remaining", y="price", color="twap_lead_bps_side",
                 color_continuous_scale="RdBu", range_color=[-15, 15], opacity=.6, facet_col="dur",
                 hover_data=["slug", "size", "z_side", "mom_5s_side"],
                 title="Buy price vs. seconds remaining (colour = how far ahead the bought side is, bps)")
fig.update_xaxes(matches=None, autorange="reversed").show()

## 8. Does it trade off a fair-value model? (price paid vs. TWAP-BM fair)

In [ ]:
fig = px.scatter(b.sample(min(len(b), 6000), random_state=0), x="fair_side", y="price", color="remaining",
                 opacity=.5, title="Price paid vs. model fair value of the bought side")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash")).show()
fig = px.histogram(b, x="edge", nbins=100, color="mkt", title="Model edge at entry (fair − price)"); fig.show()
b["edge_bin"] = pd.cut(b.edge, [-1, -.2, -.1, -.05, 0, .05, .1, .2, 1])
b.dropna(subset=["won"]).groupby("edge_bin", observed=True).agg(n=("pnl", "size"), pnl=("pnl", "sum"), win=("won", "mean"), px=("price", "mean")).round(3)

### Calibration: realised win rate vs. price paid and vs. model fair

In [ ]:
bb = b.dropna(subset=["won"]).copy()
bb["px_bin"] = (bb.price * 20).clip(0, 19.99).astype(int) / 20 + .025
bb["fair_bin"] = (bb.fair_side * 20).clip(0, 19.99).astype(int) / 20 + .025
cal = pd.concat([bb.groupby("px_bin").agg(win=("won", "mean"), n=("won", "size")).assign(kind="by price paid").rename_axis("bin"),
                 bb.groupby("fair_bin").agg(win=("won", "mean"), n=("won", "size")).assign(kind="by model fair").rename_axis("bin")]).reset_index()
fig = px.line(cal, x="bin", y="win", color="kind", markers=True, hover_data=["n"], title="Win rate of bought side")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash")).show()

## 9. Sizing

In [ ]:
fig = make_subplots(1, 3, subplot_titles=["size vs price", "size vs edge", "size vs remaining"])
s = b.sample(min(len(b), 5000), random_state=0)
for i, x in enumerate(["price", "edge", "remaining"], 1):
    fig.add_scatter(x=s[x], y=s["size"], mode="markers", marker=dict(size=4, opacity=.4), row=1, col=i, showlegend=False)
fig.update_yaxes(type="log").update_layout(height=400, title="Shares per fill").show()
print("most common share sizes:", b["size"].round(2).value_counts().head(12).to_dict())

## 10. Two-sided windows and sells (hedging / flipping / stop-outs?)

In [ ]:
w = df.groupby("slug").agg(mkt=("mkt", "first"), n=("side", "size"), sides=("outcome", "nunique"),
                           sells=("side", lambda x: (x == "SELL").sum()), pnl=("pnl", "sum"))
print(w.sides.value_counts(normalize=True).round(3))
two = df[df.slug.isin(w.index[w.sides == 2]) & (df.side == "BUY")]
vw = two.groupby(["slug", "outcome"]).apply(lambda g: np.average(g.price, weights=g["size"]), include_groups=False).unstack()
print("two-sided windows: VWAP(Up)+VWAP(Down) distribution"); print((vw.Up + vw.Down).describe().round(3))
print("first leg vs second leg: is the second leg bought after the lead flips?")
seq = two.sort_values("timestamp").groupby("slug")
def _second_leg_ahead(g):
    s = g[g.outcome != g.outcome.iloc[0]]          # empty when one side was only ever sold
    return np.nan if s.empty else float(s.twap_lead_bps_side.iloc[0] > 0)
flip = seq.apply(_second_leg_ahead, include_groups=False).dropna()
print("second-side buys made while that side is now ahead:", round(flip.mean(), 3))
sl = df[df.side == "SELL"]
print("\nSELLs: lead of the sold side at the time (bps)"); print(sl.twap_lead_bps_side.describe().round(2))
px.histogram(sl, x="twap_lead_bps_side", nbins=80, title="SELLs: TWAP lead of the side sold (bps)").show()

## 11. Rule extraction: a shallow tree for *which side* and *how much edge it demands*

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
X_cols = ["remaining", "elapsed", "twap_lead_bps_side", "z_side", "mom_3s_side", "mom_10s_side", "mom_60s_side", "vol_bps", "price", "dur"]
X = b[X_cols].fillna(0)
tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=200).fit(X.drop(columns="price"), b.price)
print("What predicts the PRICE it is willing to pay (R² = %.2f):" % tree.score(X.drop(columns="price"), b.price))
print(export_text(tree, feature_names=[c for c in X_cols if c != "price"]))
bb = b.dropna(subset=["won"])
t2 = DecisionTreeRegressor(max_depth=3, min_samples_leaf=300).fit(bb[X_cols].fillna(0), bb.pnl / bb["size"])
print("Where the per-share P&L comes from:"); print(export_text(t2, feature_names=X_cols))

## 12. One window, fill by fill

In [ ]:
SLUG = w.sort_values("n").index[-1]   # busiest window; set any slug here
g = df[df.slug == SLUG].sort_values("timestamp"); coin, t0, t1 = g.coin.iloc[0], g.t0.iloc[0], g.t1.iloc[0]
k = K[coin].loc[t0 - 60:t1 + 5]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=pd.to_datetime(k.index, unit="s"), y=k.c, name=f"{coin.upper()} Binance 1s", line=dict(color="gray"))
fig.add_hline(y=g.p0.iloc[0], line_dash="dot", annotation_text="open")
for (o, sd), gg in g.groupby(["outcome", "side"]):
    fig.add_scatter(x=gg.ts, y=gg.price, mode="markers", name=f"{sd} {o}", marker=dict(size=np.clip(gg["size"] ** .5 * 2, 5, 25),
                    symbol="triangle-up" if sd == "BUY" else "x"), secondary_y=True)
fig.update_layout(title=f"{SLUG} — resolved {'Up' if g.up_won.iloc[0] == 1 else 'Down'}", height=500).show()
g[["ts", "side", "outcome", "size", "price", "fair_side", "twap_lead_bps_side", "mom_5s_side", "remaining"]].round(3)

# 13. Backtest: the extracted rule, trading only 60c–75c
I don't have Polymarket order-book snapshots, so every **taker BUY print** in a market (from the public tape, all wallets) counts as an executable ask: at that second, someone was able to buy outcome X at price p. At each print I recompute the signal from Binance as of `LATENCY` seconds before the print, and take it if:

* `mom_3s_side > MOM_MIN`: Binance moved toward X over the last 3s
* `z_side > Z_MIN` and `fair_side − p ≥ EDGE_MIN`: X is ahead and cheap against the TWAP fair-value model
* `PX_LO ≤ p ≤ PX_HI`: **only 60c–75c**

Fill = `min(CLIP, print size)` at the print price, at most `MAX_PER_WINDOW` entries per window, with a cooldown between them. Positions are held to resolution. Fees are a taker fee of `FEE_RATE·p·(1−p)` per share; check `clob.polymarket.com/fee-rate` and the [fee docs](https://docs.polymarket.com/trading/fees) for the current rate.

Caveats: the backtest assumes you'd have won the race for that print, and print size stands in for the depth available. Treat the result as an upper bound. The latency sweep at the end shows how quickly the edge decays.

In [ ]:
BT_DAYS = 3
BT_MKTS = [(c, 5) for c in ["btc", "eth", "sol", "xrp"]] + [("btc", 15)]
PX_LO, PX_HI = 0.60, 0.75
MOM_MIN, Z_MIN, EDGE_MIN = 0.6, 0.56, 0.0      # thresholds from the tree in section 11
LATENCY = 2                                     # s between our decision and the print's block time
CLIP, MAX_PER_WINDOW, COOLDOWN = 10, 3, 10      # shares per entry, entries per window, seconds
FEE_RATE = 0.07
TAPE = CACHE.parent / "tape"; TAPE.mkdir(exist_ok=True)

In [ ]:
# every window in the backtest range, with its conditionId (cached)
MKT_F = CACHE.parent / "updown_markets.parquet"
mk = pd.read_parquet(MKT_F) if MKT_F.exists() else pd.DataFrame(columns=["slug", "conditionId", "up_won"])
end_t = min(NOW - 300, int(K["btc"].index[-1]) - 5)
slugs = [f"{c}-updown-{m}m-{t0}" for c, m in BT_MKTS
         for t0 in range((end_t - BT_DAYS * 86400) // (m * 60) * m * 60, end_t - m * 60, m * 60)]
todo = sorted(set(slugs) - set(mk.slug))
new = []
for i in range(0, len(todo), 40):
    for m_ in get_json("https://gamma-api.polymarket.com/markets", [("slug", s) for s in todo[i:i+40]] + [("limit", 100), ("closed", "true")]):
        p_ = json.loads(m_.get("outcomePrices") or "null")
        if p_ and max(map(float, p_)) == 1:
            new.append(dict(slug=m_["slug"], conditionId=m_["conditionId"], up_won=int(float(p_[0]) == 1)))
if new:
    mk = pd.concat([mk, pd.DataFrame(new)]).drop_duplicates("slug"); mk.to_parquet(MKT_F)
bt_mk = mk[mk.slug.isin(slugs)]
print(len(slugs), "windows in range,", len(bt_mk), "resolved")

In [ ]:
# taker tape per market: closed markets never change, so cache forever
from concurrent.futures import ThreadPoolExecutor
def tape(cid):
    f = TAPE / f"{cid}.parquet"
    if f.exists(): return pd.read_parquet(f)
    rows, off = [], 0
    while off < 10000:
        r = get_json("https://data-api.polymarket.com/trades", dict(market=cid, limit=500, offset=off, takerOnly="true"))
        rows += r; off += 500
        if len(r) < 500: break
    df_ = pd.DataFrame(rows, columns=["timestamp", "side", "outcome", "price", "size", "proxyWallet"])
    df_.to_parquet(f); return df_
with ThreadPoolExecutor(6) as ex:
    tapes = list(ex.map(tape, bt_mk.conditionId))
pr = pd.concat([t.assign(slug=s) for t, s in zip(tapes, bt_mk.slug)], ignore_index=True)
pr = pr.astype({"timestamp": "int64", "price": float, "size": float})
pr = pr[pr.side == "BUY"].merge(bt_mk[["slug", "up_won"]], on="slug")
m = pr.slug.str.extract(r"^(?P<coin>[a-z]+)-updown-(?P<dur>\d+)m-(?P<t0>\d{10})$")
pr = pr.join(m); pr["dur"] = pr.dur.astype(int) * 60; pr["t0"] = pr.t0.astype(int); pr["t1"] = pr.t0 + pr.dur
pr = pr[(pr.timestamp > pr.t0) & (pr.timestamp < pr.t1)]
print(f"{len(pr):,} taker-buy prints across {pr.slug.nunique():,} windows")

In [ ]:
def vstate(coin, ts, t0, t1, lat):
    k = K[coin]; arr = k.c.values; idx0 = k.index[0]
    lr = np.diff(np.log(arr), prepend=np.log(arr[0]))
    vol = pd.Series(lr).rolling(VOL_LOOKBACK, min_periods=60).std().values
    cs = np.concatenate([[0], np.cumsum(arr)])
    now = np.minimum(ts - lat, t1) - 1; i = now - idx0
    s, sig = arr[i], vol[i] * arr[i]
    R = t1 - 1 - now
    tw0 = (cs[t0 - idx0] - cs[t0 - TWAP_W - idx0]) / TWAP_W
    known = cs[i + 1] - cs[np.minimum(t1 - TWAP_W - idx0, i + 1)]
    late = R < TWAP_W
    mu = np.where(late, (known + s * R) / TWAP_W, s)
    var = sig**2 * np.where(late, R**3 / (3 * TWAP_W**2), (R - TWAP_W) + TWAP_W / 3)
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.where(var > 0, (mu - tw0) / np.sqrt(var), np.sign(mu - tw0) * 50)
    return z, ndtr(z), (s / arr[i - 3] - 1) * 1e4

def backtest(pr, lat=LATENCY, px_lo=PX_LO, px_hi=PX_HI, mom_min=MOM_MIN, z_min=Z_MIN, edge_min=EDGE_MIN):
    out = []
    for coin, g in pr.groupby("coin"):
        g = g[(g.t1 - LATENCY - 1 <= K[coin].index[-1]) & (g.t0 - TWAP_W - VOL_LOOKBACK >= K[coin].index[0])]
        z, fu, m3 = vstate(coin, g.timestamp.values, g.t0.values, g.t1.values, lat)
        sg = np.where(g.outcome == "Up", 1, -1)
        out.append(g.assign(z_side=z * sg, fair_side=np.where(sg > 0, fu, 1 - fu), mom3_side=m3 * sg))
    x = pd.concat(out)
    sig = x[(x.mom3_side > mom_min) & (x.z_side > z_min) & (x.fair_side - x.price >= edge_min) & x.price.between(px_lo, px_hi)]
    sig = sig.sort_values("timestamp")
    fills, last = [], {}
    for r in sig.itertuples():
        n, lt = last.get(r.slug, (0, -1e9))
        if n >= MAX_PER_WINDOW or r.timestamp - lt < COOLDOWN: continue
        last[r.slug] = (n + 1, r.timestamp); fills.append(r)
    f = pd.DataFrame(fills)
    if f.empty: return f
    f["qty"] = np.minimum(CLIP, f["size"])
    f["won"] = np.where(f.outcome == "Up", f.up_won, 1 - f.up_won)
    f["gross"] = f.qty * (f.won - f.price)
    f["fee"] = f.qty * FEE_RATE * f.price * (1 - f.price)
    f["net"] = f.gross - f.fee
    f["mkt"] = f.coin + "-" + (f.dur // 60).astype(str) + "m"; f["ts"] = pd.to_datetime(f.timestamp, unit="s")
    return f

bt = backtest(pr)
def summary(f):
    return pd.Series(dict(trades=len(f), windows=f.slug.nunique(), cost=(f.qty * f.price).sum(), win=f.won.mean(),
                          avg_px=f.price.mean(), gross=f.gross.sum(), fees=f.fee.sum(), net=f.net.sum(),
                          roi=f.net.sum() / (f.qty * f.price).sum()))
display(summary(bt).round(3).to_frame("all"))
bt.groupby("mkt").apply(summary, include_groups=False).round(3)

In [ ]:
bt = bt.sort_values("ts")
fig = go.Figure()
fig.add_scatter(x=bt.ts, y=bt.gross.cumsum(), name="gross")
fig.add_scatter(x=bt.ts, y=bt.net.cumsum(), name="net of taker fee")
for mk_, g in bt.groupby("mkt"):
    fig.add_scatter(x=g.ts, y=g.net.cumsum(), name=f"net {mk_}", line=dict(dash="dot"))
fig.update_layout(title=f"Backtest equity: rule, {PX_LO:.0%}–{PX_HI:.0%} only, {CLIP} shares/entry, latency {LATENCY}s", height=450).show()
bt["px_bin"] = (bt.price * 100 // 2.5 * 2.5).astype(int)
cal = bt.groupby("px_bin").agg(n=("won", "size"), win=("won", "mean"), avg_px=("price", "mean"), net=("net", "sum")).reset_index()
fig = px.bar(cal, x="px_bin", y="net", hover_data=["n", "win", "avg_px"], title="Net P&L by entry price bucket (c)"); fig.show()
cal.round(3)

### Robustness: latency, thresholds, and the wallet's own 60–75c fills

In [ ]:
sweep = []
for lat in [0, 1, 2, 3, 5]:
    for mom_min in [0.0, 0.6, 1.5]:
        f = backtest(pr, lat=lat, mom_min=mom_min)
        if len(f): sweep.append(dict(latency=lat, mom_min=mom_min, **summary(f)))
sweep = pd.DataFrame(sweep)
px.line(sweep, x="latency", y="roi", color=sweep.mom_min.astype(str), markers=True,
        title="Net ROI vs. latency (colour = MOM_MIN bps)").show()
sweep.round(3)

In [ ]:
w60 = df[(df.side == "BUY") & df.price.between(PX_LO, PX_HI) & (df.timestamp >= int(pr.t0.min()))]
w60 = w60.assign(fee=w60["size"] * FEE_RATE * w60.price * (1 - w60.price))
print(f"wallet's actual 60–75c buys in the same period: {len(w60)} fills, cost ${w60.usd.sum():,.0f}, "
      f"P&L ${w60.pnl.sum():,.0f} gross / ${(w60.pnl - w60.fee).sum():,.0f} after taker fee (upper bound; some fills are maker), "
      f"win {w60.won.mean():.3f} vs avg px {w60.price.mean():.3f}")

### Backtest results (run of 2026-10-29, 3 days, 3,744 windows, 1.48M taker prints)
| | trades | cost | win | avg px | gross | taker fee | **net** | ROI |
|---|---|---|---|---|---|---|---|---|
| rule, 60–75c, latency 2s | 3,583 | $17.2k | 74.2% | 0.699 | +$1,101 | −$360 | **+$740** | **+4.3%** |

* **Per market:** BTC/ETH/SOL/XRP 5m all net positive (+4% to +8%). **BTC 15m loses (−2.4%)**, so drop it.
* **Latency decides everything.** Net ROI at mom ≥ 0.6 bps: 8.8% at 0s, 5.5% at 1s, 4.3% at 2s, 0.8% at 3s, −1.6% at 5s. Beyond about 3s from the Binance tick to on-chain inclusion, the edge is gone.
* **The momentum filter matters.** With no momentum filter (pure fair value), ROI roughly halves at every latency.
* **For comparison, the wallet's own 60–75c buys lost money** over the same 3 days (−$460 gross, 68.7% win vs. 67.7% avg price). Its edge sits outside this band, or comes from fills this rule would have skipped.
* The fill assumption is optimistic: it assumes we win every print race and can take up to the print size. Read these results as an upper bound.

# 14. Does it carry over to Kalshi `KXBTC15M`?
Kalshi's BTC 15m market settles the same way: Yes if the 60s average of CF Benchmarks' **BRTI** before close is ≥ the 60s average before open. So the TWAP fair-value model carries over unchanged. Differences:
* **Reference index**: BRTI is a composite of Coinbase, Kraken, Bitstamp and others, and **Binance isn't in it**. Our Binance signal is therefore a step removed from what Kalshi settles on.
* **Fees**: Kalshi's taker fee is `ceil(0.07 · C · P · (1−P))` cents per order, the same curve as assumed above, rounded up per order.
* **Fills**: the tape has every print with `taker_side` and millisecond timestamps. A taker buy of YES at p is an executable Up ask at p, and a taker buy of NO is an executable Down ask.

Same rule, same 60–75c band, same latency.

In [ ]:
KB = "https://api.elections.kalshi.com/trade-api/v2"
KAL = CACHE.parent / "kalshi"; KAL.mkdir(exist_ok=True)
KAL_MKT_F = KAL / "KXBTC15M_markets.parquet"

def kget(path, params):
    for i in range(20):
        time.sleep(0.12)   # stay under Kalshi's basic-tier read limit
        try: r = S.get(KB + path, params=params, timeout=30)
        except requests.RequestException: time.sleep(min(60, 0.5 * 2**i)); continue
        if r.status_code == 429 or r.status_code >= 500: time.sleep(min(60, 0.5 * 2**i)); continue
        r.raise_for_status(); return r.json()
    raise RuntimeError(path)

km = pd.read_parquet(KAL_MKT_F) if KAL_MKT_F.exists() else pd.DataFrame()
if km.empty or km.close_ts.max() < end_t - 1800:
    rows, cur = [], None
    while True:
        p = dict(series_ticker="KXBTC15M", status="settled", limit=1000, min_close_ts=end_t - BT_DAYS * 86400, max_close_ts=end_t)
        if cur: p["cursor"] = cur
        r = kget("/markets", p); rows += r["markets"]; cur = r.get("cursor")
        if not cur or not r["markets"]: break
    new = pd.DataFrame([dict(ticker=m["ticker"], t0=int(pd.Timestamp(m["open_time"]).timestamp()),
                             close_ts=int(pd.Timestamp(m["close_time"]).timestamp()), result=m["result"]) for m in rows])
    km = pd.concat([km, new]).drop_duplicates("ticker"); km.to_parquet(KAL_MKT_F)
km = km[(km.close_ts <= end_t) & (km.close_ts > end_t - BT_DAYS * 86400) & km.result.isin(["yes", "no"])]
print(len(km), "settled KXBTC15M markets")

In [ ]:
def ktape(ticker):
    f = KAL / f"{ticker}.parquet"
    if f.exists(): return pd.read_parquet(f)
    rows, cur = [], None
    while True:
        p = dict(ticker=ticker, limit=1000)
        if cur: p["cursor"] = cur
        r = kget("/markets/trades", p); rows += r["trades"]; cur = r.get("cursor")
        if not cur or not r["trades"]: break
    d_ = pd.DataFrame(rows, columns=["created_time", "taker_side", "yes_price_dollars", "no_price_dollars", "count_fp"])
    d_.to_parquet(f); return d_
def ktape_safe(ticker):
    try: return ktape(ticker)
    except RuntimeError: return None      # not cached, so the next run retries it
with ThreadPoolExecutor(2) as ex:
    kt = list(ex.map(ktape_safe, km.ticker))
print(f"tapes: {sum(t is not None for t in kt)}/{len(kt)} markets (rerun to fill any gaps)")
kp = pd.concat([t.assign(ticker=k) for t, k in zip(kt, km.ticker) if t is not None], ignore_index=True).merge(km, on="ticker")
kp["timestamp"] = (pd.to_datetime(kp.created_time, format="ISO8601").astype("int64") // 10**9).astype("int64")
kp["outcome"] = np.where(kp.taker_side == "yes", "Up", "Down")
kp["price"] = np.where(kp.taker_side == "yes", kp.yes_price_dollars.astype(float), kp.no_price_dollars.astype(float))
kp["size"] = kp.count_fp.astype(float)
kp["up_won"] = (kp.result == "yes").astype(int)
kp = kp.assign(coin="btc", dur=900, t1=kp.close_ts, slug=kp.ticker)
kp = kp[(kp.timestamp > kp.t0) & (kp.timestamp < kp.t1)]
print(f"{len(kp):,} taker prints across {kp.ticker.nunique()} markets")

In [ ]:
# how well does Binance stand in for BRTI on settlement?
k0 = km.assign(t1=km.close_ts)
kk = K["btc"]; c_ = kk.c
ok_ = (k0.t0 - TWAP_W >= kk.index[0]) & (k0.t1 <= kk.index[-1])
k0 = k0[ok_]
tw = lambda a, b: np.array([c_.loc[x:y - 1].mean() for x, y in zip(a, b)])
bin_up = (tw(k0.t1 - TWAP_W, k0.t1) >= tw(k0.t0 - TWAP_W, k0.t0)).astype(int)
print(f"Binance 60s-avg proxy matches Kalshi settlement on {(bin_up == (k0.result == 'yes')).mean():.1%} of {len(k0)} markets")

In [ ]:
kbt = backtest(kp)
# Kalshi rounds the taker fee up to the cent per order
kbt["fee"] = np.ceil(FEE_RATE * kbt.qty * kbt.price * (1 - kbt.price) * 100 - 1e-9) / 100
kbt["net"] = kbt.gross - kbt.fee
pbtc = bt[bt.mkt == "btc-15m"]
pd.DataFrame({"Kalshi KXBTC15M": summary(kbt), "Polymarket btc-15m": summary(pbtc)}).round(3)

In [ ]:
ksw = []
for lat in [0, 1, 2, 3, 5]:
    for mom_min in [0.0, 0.6, 1.5]:
        f = backtest(kp, lat=lat, mom_min=mom_min)
        if len(f):
            f["fee"] = np.ceil(FEE_RATE * f.qty * f.price * (1 - f.price) * 100 - 1e-9) / 100; f["net"] = f.gross - f.fee
            ksw.append(dict(latency=lat, mom_min=mom_min, **summary(f)))
ksw = pd.DataFrame(ksw)
px.line(ksw, x="latency", y="roi", color=ksw.mom_min.astype(str), markers=True, title="Kalshi KXBTC15M: net ROI vs latency (colour = MOM_MIN)").show()
kbt = kbt.sort_values("ts"); kbt["day"] = kbt.ts.dt.strftime("%Y-%m-%d")
display(kbt.groupby("day").agg(trades=("net", "size"), gross=("gross", "sum"), fee=("fee", "sum"), net=("net", "sum")).round(2))
ksw.round(3)